In [7]:
!pip install -q transformers accelerate bitsandbytes tqdm

import glob
import json
import re
import string
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm

# 1. Metrics
def normalize_answer(s):
    def remove_articles(text): return re.sub(r'\b(a|an|the)\b', ' ', text)
    def white_space_fix(text): return ' '.join(text.split())
    def remove_punc(text): return ''.join(ch for ch in text if ch not in set(string.punctuation))
    def lower(text): return text.lower()
    return white_space_fix(remove_articles(remove_punc(lower(s))))

def compute_exact(a_gold, a_pred):
    return int(normalize_answer(a_gold) == normalize_answer(a_pred))

def compute_f1(a_gold, a_pred):
    gold_toks = normalize_answer(a_gold).split()
    pred_toks = normalize_answer(a_pred).split()
    common = set(gold_toks) & set(pred_toks)
    num_same = sum(min(gold_toks.count(tok), pred_toks.count(tok)) for tok in common)
    if len(gold_toks) == 0 or len(pred_toks) == 0: return int(gold_toks == pred_toks)
    if num_same == 0: return 0.0
    precision = 1.0 * num_same / len(pred_toks)
    recall = 1.0 * num_same / len(gold_toks)
    return (2 * precision * recall) / (precision + recall)

# 2. Load dataset
json_paths = glob.glob("/kaggle/input/**/squad_top5_for_colab.json", recursive=True)
if not json_paths:
    raise FileNotFoundError("File not found in /kaggle/input/. Check dataset upload.")

data_file = json_paths[0]
print(f"Loading data from: {data_file}")
with open(data_file, "r", encoding="utf-8") as f:
    records = json.load(f)

# 3. Load FLAN-T5-XXL (11B) in float16 across 2x T4 GPUs (30GB total VRAM)
model_id = "google/flan-t5-xxl"
print(f"Loading {model_id} across 2x T4 GPUs...")

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)
print("Model loaded successfully across both GPUs.")

# 4. Generate
em_scores, f1_scores = [], []
print(f"Running Generation on {len(records)} questions...")

for item in tqdm(records):
    question = item["question"]
    context = "\n".join([p["text"] for p in item["retrieved_top5"][:3]])
    prompt = f"Answer the following question based on the context.\n\nContext:\n{context}\n\nQuestion: {question}\nAnswer:"

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=32)
    pred_answer = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

    golds = item["ground_truth_answers"]
    em = max((compute_exact(g, pred_answer) for g in golds), default=0)
    f1 = max((compute_f1(g, pred_answer) for g in golds), default=0.0)
    em_scores.append(em)
    f1_scores.append(f1)

mean_em = (sum(em_scores) / len(em_scores)) * 100
mean_f1 = (sum(f1_scores) / len(f1_scores)) * 100

print("\n" + "="*50)
print(f"FLAN-T5-XXL (11B) SQuAD Generation Results:")
print(f"Exact Match (EM) : {mean_em:.2f}% (Paper: 57.63%)")
print(f"Token F1 Score   : {mean_f1:.2f}% (Paper: 68.40%)")
print("="*50)

Loading data from: /kaggle/input/datasets/kira63004/squad-top5/squad_top5_for_colab.json
Loading google/flan-t5-xxl across 2x T4 GPUs...


Loading weights:   0%|          | 0/559 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [1]:
!pip install -q beir ranx rank-bm25 faiss-cpu transformers accelerate bitsandbytes sentence-transformers scipy numpy tqdm

import os
import gc
import re
import string
import time
from collections import defaultdict
import numpy as np
import scipy.sparse as sp
import torch
from beir import util
from beir.datasets.data_loader import GenericDataLoader
from ranx import Qrels, Run, evaluate
from rank_bm25 import BM25Okapi
import faiss
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForMaskedLM, AutoModelForSeq2SeqLM
from tqdm import tqdm

# Free any stray allocations
gc.collect()
torch.cuda.empty_cache()

# ==========================================
# 1. Download & Load BEIR NQ Dataset
# ==========================================
dataset = "nq"
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{dataset}.zip"
out_dir = "/kaggle/working/datasets"
data_path = util.download_and_unzip(url, out_dir)

print("Loading NQ corpus, queries, and qrels...")
corpus, queries, qrels = GenericDataLoader(data_folder=data_path).load(split="test")

eval_qids = list(queries.keys())[:500]
eval_queries = {qid: queries[qid] for qid in eval_qids}
eval_qrels = {qid: qrels[qid] for qid in eval_qids}

gold_dids = set()
for qid in eval_qids:
    gold_dids.update(eval_qrels[qid].keys())

pool_corpus = {}
for did, item in corpus.items():
    if did in gold_dids or len(pool_corpus) < 15000:
        pool_corpus[did] = item

doc_ids = list(pool_corpus.keys())
print(f"Loaded NQ: {len(pool_corpus)} passages | {len(eval_queries)} queries.")

# ==========================================
# 2. Retriever 1: BM25
# ==========================================
print("\n--- Running BM25 ---")
def tokenize(text): return re.findall(r"\w+", text.lower())

bm25_corpus = [tokenize(f"{pool_corpus[d]['title']} {pool_corpus[d]['text']}") for d in doc_ids]
bm25 = BM25Okapi(bm25_corpus)

bm25_runs = {}
for qid, q in tqdm(eval_queries.items(), desc="BM25"):
    scores = bm25.get_scores(tokenize(q))
    top_idx = np.argsort(scores)[-50:][::-1]
    bm25_runs[qid] = {doc_ids[i]: float(scores[i]) for i in top_idx}

# ==========================================
# 3. Retriever 2: Dense MiniLM on GPU 0
# ==========================================
print("\n--- Running Dense MiniLM ---")
dense_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda:0")
doc_texts = [f"{pool_corpus[d]['title']} {pool_corpus[d]['text']}".strip() for d in doc_ids]
doc_embs = dense_model.encode(doc_texts, batch_size=128, show_progress_bar=True, normalize_embeddings=True)

dim = doc_embs.shape[1]
faiss_idx = faiss.IndexFlatIP(dim)
faiss_idx.add(np.array(doc_embs, dtype="float32"))

dense_runs = {}
for qid, q in tqdm(eval_queries.items(), desc="Dense"):
    q_emb = dense_model.encode([q], normalize_embeddings=True).astype("float32")
    scores, indices = faiss_idx.search(q_emb, 50)
    dense_runs[qid] = {doc_ids[idx]: float(scores[0][i]) for i, idx in enumerate(indices[0])}

# Free dense memory before SPLADE
del dense_model, doc_embs
gc.collect()
torch.cuda.empty_cache()

# ==========================================
# 4. Retriever 3: SPLADE in FP16 on GPU 0
# ==========================================
print("\n--- Running SPLADE in float16 ---")
splade_device = "cuda:0"
splade_tok = AutoTokenizer.from_pretrained("naver/splade-cocondenser-ensembledistil")
splade_model = AutoModelForMaskedLM.from_pretrained(
    "naver/splade-cocondenser-ensembledistil",
    torch_dtype=torch.float16
).to(splade_device)
splade_model.eval()

def encode_splade_batch(texts, batch_size=16):
    rows, cols, data = [], [], []
    for i in tqdm(range(0, len(texts), batch_size), desc="Encoding SPLADE"):
        b_texts = texts[i:i+batch_size]
        toks = splade_tok(b_texts, return_tensors="pt", padding=True, truncation=True, max_length=256).to(splade_device)
        with torch.no_grad():
            logits = splade_model(**toks).logits
            relu_log = torch.log1p(torch.relu(logits))
            mask = toks["attention_mask"].unsqueeze(-1)
            reps, _ = torch.max(relu_log * mask, dim=1)
        reps = reps.cpu().numpy()
        for r_idx, rep in enumerate(reps):
            g_idx = i + r_idx
            nonzero = np.nonzero(rep)[0]
            rows.extend([g_idx] * len(nonzero))
            cols.extend(nonzero)
            data.extend(rep[nonzero])
    return sp.csr_matrix((data, (rows, cols)), shape=(len(texts), splade_tok.vocab_size), dtype=np.float32)

splade_doc_matrix = encode_splade_batch(doc_texts, batch_size=16)

splade_runs = {}
for qid, q in tqdm(eval_queries.items(), desc="SPLADE"):
    toks = splade_tok(q, return_tensors="pt", truncation=True, max_length=128).to(splade_device)
    with torch.no_grad():
        out = splade_model(**toks).logits
        q_rep, _ = torch.max(torch.log1p(torch.relu(out)) * toks["attention_mask"].unsqueeze(-1), dim=1)
    q_vec = q_rep.squeeze(0).cpu().numpy()
    scores = splade_doc_matrix.dot(q_vec)
    top_idx = np.argsort(scores)[-50:][::-1]
    splade_runs[qid] = {doc_ids[i]: float(scores[i]) for i in top_idx}

# Free SPLADE memory before loading FLAN-T5-XXL
del splade_model, splade_doc_matrix
gc.collect()
torch.cuda.empty_cache()

# ==========================================
# 5. 3-Way Reciprocal Rank Fusion (Blended RAG)
# ==========================================
print("\n--- Fusing with 3-Way RRF (Blended RAG) ---")
blended_runs = {}
for qid in eval_queries:
    fused = defaultdict(float)
    for rank, (d, _) in enumerate(bm25_runs[qid].items(), start=1):
        fused[d] += 1.0 / (60 + rank)
    for rank, (d, _) in enumerate(dense_runs[qid].items(), start=1):
        fused[d] += 1.0 / (60 + rank)
    for rank, (d, _) in enumerate(splade_runs[qid].items(), start=1):
        fused[d] += 1.0 / (60 + rank)
    sorted_docs = sorted(fused.items(), key=lambda x: x[1], reverse=True)[:50]
    blended_runs[qid] = {d: s for d, s in sorted_docs}

# Evaluations
ranx_qrels = Qrels(eval_qrels)
m_bm25 = evaluate(ranx_qrels, Run(bm25_runs), metrics=["hit_rate@5", "hit_rate@20", "ndcg@10"])
m_dense = evaluate(ranx_qrels, Run(dense_runs), metrics=["hit_rate@5", "hit_rate@20", "ndcg@10"])
m_splade = evaluate(ranx_qrels, Run(splade_runs), metrics=["hit_rate@5", "hit_rate@20", "ndcg@10"])
m_blended = evaluate(ranx_qrels, Run(blended_runs), metrics=["hit_rate@5", "hit_rate@20", "ndcg@10"])

print("\n" + "="*75)
print(f" NQ ABLATION: INDIVIDUAL RETRIEVERS VS. BLENDED RAG")
print("="*75)
print(f"{'Method':<25} | {'Top-5 Acc':<12} | {'Top-20 Acc':<12} | {'NDCG@10':<12}")
print("-" * 75)
print(f"{'BM25 (Single)':<25} | {m_bm25['hit_rate@5']*100:>10.2f}% | {m_bm25['hit_rate@20']*100:>10.2f}% | {m_bm25['ndcg@10']:>10.4f}")
print(f"{'Dense (MiniLM)':<25} | {m_dense['hit_rate@5']*100:>10.2f}% | {m_dense['hit_rate@20']*100:>10.2f}% | {m_dense['ndcg@10']:>10.4f}")
print(f"{'SPLADE (Sparse)':<25} | {m_splade['hit_rate@5']*100:>10.2f}% | {m_splade['hit_rate@20']*100:>10.2f}% | {m_splade['ndcg@10']:>10.4f}")
print(f"{'Blended RAG (3-Way RRF)':<25} | {m_blended['hit_rate@5']*100:>10.2f}% | {m_blended['hit_rate@20']*100:>10.2f}% | {m_blended['ndcg@10']:>10.4f}")
print("="*75)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.3/99.3 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 85.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 46.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 462.5/462.5 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 63.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.8/947.8 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.9/304.9 kB 17.8 MB/s eta 0:00:00


/usr/local/lib/python3.13/dist-packages/beir/util.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


/kaggle/working/datasets/nq.zip:   0%|          | 0.00/475M [00:00<?, ?iB/s]

Loading NQ corpus, queries, and qrels...


  0%|          | 0/2681468 [00:00<?, ?it/s]

...

model.safetensors: downloading bytes:           |  0.00B            



Encoding SPLADE:   0%|          | 0/945 [00:00<?, ?it/s]

Encoding SPLADE:   0%|          | 1/945 [00:00<07:08,  2.20it/s]

Encoding SPLADE:   0%|          | 3/945 [00:00<02:39,  5.90it/s]

Encoding SPLADE:   1%|          | 5/945 [00:00<01:52,  8.32it/s]

Encoding SPLADE:   1%|          | 7/945 [00:00<01:30, 10.37it/s]

Encoding SPLADE:   1%|          | 9/945 [00:00<01:17, 12.10it/s]

Encoding SPLADE:   1%|          | 11/945 [00:01<01:13, 12.74it/s]

Encoding SPLADE:   1%|▏         | 13/945 [00:01<01:06, 14.04it/s]

Encoding SPLADE:   2%|▏         | 15/945 [00:01<01:04, 14.43it/s]

Encoding SPLADE:   2%|▏         | 17/945 [00:01<01:03, 14.52it/s]

Encoding SPLADE:   2%|▏         | 19/945 [00:01<01:00, 15.38it/s]

Encoding SPLADE:   2%|▏         | 21/945 [00:01<01:01, 14.93it/s]

Encoding SPLADE:   2%|▏         | 23/945 [00:01<01:02, 14.69it/s]

Encoding SPLADE:   3%|▎         | 25/945 [00:02<01:02, 14.72it/s]

Encoding SPLADE:   3%|▎         | 27/945 [00:02<01:03, 14.46it/s]

Encodin


--- Fusing with 3-Way RRF (Blended RAG) ---


/usr/local/lib/python3.13/dist-packages/ranx/metrics/hit_rate.py:38: NumbaTypeSafetyWarning: unsafe cast from uint64 to int64. Precision may be lost.
  scores[i] = _hit_rate(qrels[i], run[i], k, rel_lvl)



 NQ ABLATION: INDIVIDUAL RETRIEVERS VS. BLENDED RAG
Method                    | Top-5 Acc    | Top-20 Acc   | NDCG@10     
---------------------------------------------------------------------------
BM25 (Single)             |      67.00% |      84.40% |     0.5433
Dense (MiniLM)            |      80.60% |      96.00% |     0.6751
SPLADE (Sparse)           |      89.00% |      98.20% |     0.7589
Blended RAG (3-Way RRF)   |      83.00% |      97.20% |     0.7094


In [11]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()